In [ ]:
# 1. IMPORT LIBRARY
import os, re, sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

warnings.filterwarnings('ignore')
print('Library berhasil diimport!')


In [ ]:
# Lazy init EasyOCR
reader = None

def get_ocr_reader():
    global reader
    if reader is None:
        try:
            import easyocr
            reader = easyocr.Reader(['id', 'en'], gpu=False)
        except ImportError:
            reader = False
    return reader

def ocr_image(img_np):
    rdr = get_ocr_reader()
    if not rdr: return ''
    try:
        results = rdr.readtext(img_np, detail=0)
        return ' '.join(results)
    except:
        return ''

def extract_text_from_file(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ''
    try:
        if ext == '.pdf':
            doc = fitz.open(file_path)
            for page in doc:
                page_text = page.get_text().strip()
                if page_text:
                    text += page_text + '\n'
                else:
                    import cv2
                    pix = page.get_pixmap(dpi=200)
                    img = cv2.imdecode(np.frombuffer(pix.tobytes('png'), np.uint8), cv2.IMREAD_COLOR)
                    if img is not None:
                        ocr = ocr_image(img)
                        if ocr: text += ocr + '\n'
            doc.close()
        elif ext in {'.jpg','.jpeg','.png','.bmp','.tiff','.tif'}:
            import cv2
            img = cv2.imread(file_path)
            if img is not None: text += ocr_image(img) + '\n'
    except Exception as e:
        print(f'  [ERROR] {os.path.basename(file_path)}: {e}')
    return text.strip()

print('Fungsi ekstraksi teks siap.')


In [ ]:
# 2. Baca File Dataset
DATASET_DIR = Path('./dataset')
if not DATASET_DIR.exists():
    DATASET_DIR = Path('../dataset')

print(f'Dataset: {DATASET_DIR.resolve()}')

JENIS_KE_ARAH = {
    'PurchaseOrder': 'Masuk',
    'Invoice': 'Keluar',
    'Penawaran': 'Keluar',
    'SalesOrder': 'Keluar',
    'SuratJalan': 'Keluar',
}

data = []
for folder_name in sorted(os.listdir(DATASET_DIR)):
    folder_path = DATASET_DIR / folder_name
    if not folder_path.is_dir(): continue
    jenis = folder_name
    arah = JENIS_KE_ARAH.get(jenis, 'Keluar')
    files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    print(f'Folder {folder_name}: {len(files)} file')
    for f in sorted(files):
        text = extract_text_from_file(str(folder_path / f))
        if text:
            data.append({'file': f, 'text': text, 'arah': arah, 'jenis': jenis})
            print(f'  [OK] {f}')
        else:
            print(f'  [SKIP] {f}')

df = pd.DataFrame(data)
print(f'\nTotal: {len(df)} dokumen')
print(df['jenis'].value_counts())


In [ ]:
# 3. Validasi Data
print('Validasi:')
for jenis, count in df['jenis'].value_counts().items():
    status = 'OK' if count >= 5 else 'KURANG'
    print(f'  {jenis}: {count} [{status}]')
print(f'\nTotal: {len(df)} dokumen')


In [ ]:
# 4. Preprocessing
stemmer = StemmerFactory().create_stemmer()
stopwords = set(StopWordRemoverFactory().get_stop_words())
company_sw = {'pt','cv','tbk','abt','vi','nomor','perihal','lampiran','kepada','yth'}
domain_sw = {'rucika','pcs','batang','total','harga','diskon','tanggal','kode',
    'barang','nama','qty','satuan','rupiah','ribu','juta','indonesia',
    'tangerang','banten','kota','green','lake','city','ruko','timur',
    'bintang','almex','jan','feb','mar','apr','mei','jun','jul','agu',
    'sep','okt','nov','des','no','jumlah','sub','lain','biaya','ppn',
    'dpp','net','cash','transfer','dibuat','disetujui','pengirim',
    'penerima','keterangan'}
all_sw = stopwords | company_sw | domain_sw

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\b\d{4}\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in all_sw and len(t) > 2]
    if tokens:
        tokens = stemmer.stem(' '.join(tokens)).split()
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)
df = df[df['clean_text'] != ''].reset_index(drop=True)
print(f'After cleaning: {len(df)}')


In [ ]:
# 5. Split Data
X = df['clean_text']
y_arah = df['arah']
y_jenis = df['jenis']

X_train, X_test, ytr_a, yte_a, ytr_j, yte_j = train_test_split(
    X, y_arah, y_jenis, test_size=0.2, random_state=42, stratify=y_jenis)

print(f'Train: {len(X_train)}, Test: {len(X_test)}')


In [ ]:
# 6. TF-IDF Params
n_docs = len(df)
max_features = min(800, max(300, n_docs * 8))
min_df = 3 if n_docs >= 80 else 2
max_df = 0.70
print(f'max_features={max_features}, min_df={min_df}, max_df={max_df}')


In [ ]:
# 7. Training Model Arah (PIPELINE)
pipe_arah = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1,2),
                               min_df=min_df, max_df=max_df, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0))
])

# Fit di FULL dataset
pipe_arah.fit(df['clean_text'], y_arah)

# Evaluasi di test set
pred_a = pipe_arah.predict(X_test)
print('ARAH:')
print(f'  Acc: {accuracy_score(yte_a, pred_a):.4f}')
print(f'  F1 : {f1_score(yte_a, pred_a, average="weighted"):.4f}')
print(classification_report(yte_a, pred_a))


In [ ]:
# 8. Confusion Matrix Arah
cm = confusion_matrix(yte_a, pred_a, labels=pipe_arah.classes_)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=pipe_arah.classes_, yticklabels=pipe_arah.classes_)
plt.title('CM - Arah Dokumen')
plt.tight_layout()
plt.savefig('cm_arah.png', dpi=150)
plt.show()


In [ ]:
# 9. Training Model Jenis (PIPELINE)
pipe_jenis = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=max_features, ngram_range=(1,2),
                               min_df=min_df, max_df=max_df, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0))
])

# Fit di FULL dataset
pipe_jenis.fit(df['clean_text'], y_jenis)

# Evaluasi di test set
pred_j = pipe_jenis.predict(X_test)
print('JENIS:')
print(f'  Acc: {accuracy_score(yte_j, pred_j):.4f}')
print(f'  F1 : {f1_score(yte_j, pred_j, average="weighted"):.4f}')
print(classification_report(yte_j, pred_j))


In [ ]:
# 10. Confusion Matrix Jenis
cm = confusion_matrix(yte_j, pred_j, labels=pipe_jenis.classes_)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=pipe_jenis.classes_, yticklabels=pipe_jenis.classes_)
plt.title('CM - Jenis Dokumen')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('cm_jenis.png', dpi=150)
plt.show()


In [ ]:
# 11. Cross-Validation
n_folds = min(5, max(2, min(y_jenis.value_counts().min(), y_arah.value_counts().min())))

cv_a = cross_val_score(pipe_arah, df['clean_text'], y_arah,
    cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42), scoring='accuracy')
cv_j = cross_val_score(pipe_jenis, df['clean_text'], y_jenis,
    cv=StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42), scoring='accuracy')

print(f'CV Arah : Mean={cv_a.mean():.4f} Std={cv_a.std():.4f}')
print(f'CV Jenis: Mean={cv_j.mean():.4f} Std={cv_j.std():.4f}')

fig, ax = plt.subplots(1,2, figsize=(12,5))
ax[0].bar(range(1,n_folds+1), cv_a); ax[0].set_title('CV Arah')
ax[1].bar(range(1,n_folds+1), cv_j); ax[1].set_title('CV Jenis')
plt.savefig('cv.png', dpi=150)
plt.show()


In [ ]:
# 12. Simpan Model (PIPELINE)
import joblib
MODEL_DIR = Path('../backend/ml_model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(pipe_arah, MODEL_DIR / 'arah_pipeline.pkl')
joblib.dump(pipe_jenis, MODEL_DIR / 'jenis_pipeline.pkl')

# Hapus tfidf lama
old = MODEL_DIR / 'tfidf_vectorizer.pkl'
if old.exists():
    old.unlink()
    print('Hapus tfidf_vectorizer.pkl lama')

print('Model tersimpan!')
print(f'  -> {MODEL_DIR}')


In [ ]:
# 13. Test Manual
test = input('Masukkan teks (kosong=skip): ').strip()
if test:
    a = pipe_arah.predict([test])[0]
    j = pipe_jenis.predict([test])[0]
    ac = max(pipe_arah.predict_proba([test])[0])
    jc = max(pipe_jenis.predict_proba([test])[0])
    print(f'Arah : {a} ({ac:.1%})')
    print(f'Jenis: {j} ({jc:.1%})')


In [ ]:
# 14. ROBUSTNESS TESTING - Uji Ketahanan Model
import random
random.seed(42)

# Pastikan pipeline sudah di-fit pada training set
pipe_arah.fit(X_train_text, y_train_arah)
pipe_jenis.fit(X_train_text, y_train_jenis)

def delete_keywords(text, pct=0.30):
    """Hapus x% token secara acak (simulasi informasi parsial hilang)"""
    tokens = text.split()
    if len(tokens) < 5:
        return text
    n_drop = max(1, int(len(tokens) * pct))
    keep_indices = sorted(random.sample(range(len(tokens)), len(tokens) - n_drop))
    keep = [tokens[i] for i in keep_indices]
    return ' '.join(keep)

def ocr_noise(text, pct=0.20):
    """Simulasi error OCR: ganti x% karakter dengan typo"""
    chars = list(text)
    n_change = max(1, int(len(chars) * pct))
    indices = random.sample(range(len(chars)), min(n_change, len(chars)))

    # Mapping typo: huruf kecil, besar, angka, simbol
    nearby = {
        'a':'s', 's':'a', 'i':'1', 'o':'0', 'e':'3', 't':'7', 'n':'m', 'm':'n',
        'r':'t', 'u':'v', 'k':'l', 'l':'1', 'b':'6', 'g':'9', 'q':'9', 'd':'o',
        'A':'S', 'S':'A', 'I':'1', 'O':'0', 'E':'3', 'T':'7', 'N':'M', 'M':'N',
        'R':'T', 'U':'V', 'K':'L', 'B':'8', 'G':'6', 'Q':'0', 'D':'0',
        '0':'O', '1':'I', '2':'Z', '3':'E', '5':'S', '6':'G', '7':'T', '8':'B', '9':'g',
        '.':',', ',':'.', '-':'_', '/':'|', ':':';', ';':':'
    }
    for i in indices:
        if chars[i] in nearby:
            chars[i] = nearby[chars[i]]
        elif chars[i].isalpha():
            chars[i] = chars[i].swapcase()
    return ''.join(chars)

results = []

# Baseline (tanpa gangguan)
pred_base_a = pipe_arah.predict(X_test_text)
pred_base_j = pipe_jenis.predict(X_test_text)
f1_base_a = f1_score(y_test_arah, pred_base_a, average='weighted')
f1_base_j = f1_score(y_test_jenis, pred_base_j, average='weighted')
results.append({
    'Skenario': 'Tanpa Gangguan (Baseline)',
    'Parameter': '-',
    'F1 Arah': f1_base_a,
    'F1 Jenis': f1_base_j,
    'Keterangan': 'Model tanpa modifikasi input'
})

# 1. Keyword Deletion 30%
X_kw = [delete_keywords(t, 0.30) for t in X_test_text]
pred_kw_a = pipe_arah.predict(X_kw)
pred_kw_j = pipe_jenis.predict(X_kw)
f1_kw_a = f1_score(y_test_arah, pred_kw_a, average='weighted')
f1_kw_j = f1_score(y_test_jenis, pred_kw_j, average='weighted')
results.append({
    'Skenario': 'Hapus Keyword',
    'Parameter': '30% token',
    'F1 Arah': f1_kw_a,
    'F1 Jenis': f1_kw_j,
    'Keterangan': 'Simulasi informasi parsial hilang'
})

# 2. OCR Noise 20%
X_ocr = [ocr_noise(t, 0.20) for t in X_test_text]
pred_ocr_a = pipe_arah.predict(X_ocr)
pred_ocr_j = pipe_jenis.predict(X_ocr)
f1_ocr_a = f1_score(y_test_arah, pred_ocr_a, average='weighted')
f1_ocr_j = f1_score(y_test_jenis, pred_ocr_j, average='weighted')
results.append({
    'Skenario': 'OCR Error',
    'Parameter': '20% karakter',
    'Keterangan': 'Simulasi hasil scan blur/kotor'
})

# Tabel hasil
df_robust = pd.DataFrame(results)
print('HASIL ROBUSTNESS TESTING')
print('=' * 70)
for _, row in df_robust.iterrows():
    print(f"{row['Skenario']:30s} | Arah: {row['F1 Arah']:.4f} | Jenis: {row['F1 Jenis']:.4f}")

# Visualisasi
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x_labels = df_robust['Skenario']

ax[0].bar(range(len(x_labels)), df_robust['F1 Arah'], color=['#22c55e', '#f59e0b', '#ef4444'])
ax[0].set_xticks(range(len(x_labels)))
ax[0].set_xticklabels(x_labels, rotation=15, ha='right')
ax[0].set_ylim(0, 1.05)
ax[0].set_ylabel('F1 Score')
ax[0].set_title('Robustness - Klasifikasi Arah')
for i, v in enumerate(df_robust['F1 Arah']):
    ax[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

ax[1].bar(range(len(x_labels)), df_robust['F1 Jenis'], color=['#22c55e', '#f59e0b', '#ef4444'])
ax[1].set_xticks(range(len(x_labels)))
ax[1].set_xticklabels(x_labels, rotation=15, ha='right')
ax[1].set_ylim(0, 1.05)
ax[1].set_ylabel('F1 Score')
ax[1].set_title('Robustness - Klasifikasi Jenis')
for i, v in enumerate(df_robust['F1 Jenis']):
    ax[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: robustness.png')
